# Cross-Chain DeFi Environment Demo

This notebook demonstrates the multi-chain DeFi simulator environment.

## Overview

The environment models:
- Multiple blockchain networks (Ethereum, Arbitrum, etc.)
- Cross-chain bridges with latency and failure risk
- DEX liquidity pools with AMM mechanics
- Gas costs and transaction execution

In [ ]:
import sys

sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
from envs.cross_chain_env import CrossChainEnv, Chain, Bridge, Pool

## Create Environment

In [ ]:
# Define chains
chains = [
    Chain(
        name="Ethereum",
        chain_id=1,
        block_time=12.0,
        base_gas_price=20.0,
        gas_volatility=0.1,
    ),
    Chain(
        name="Arbitrum",
        chain_id=42161,
        block_time=0.25,
        base_gas_price=0.1,
        gas_volatility=0.05,
    ),
]

# Define bridges
bridges = [
    Bridge(
        name="Arbitrum Bridge",
        source_chain="Ethereum",
        target_chain="Arbitrum",
        capacity=10000.0,
        latency_mean=600.0,
        latency_std=120.0,
        failure_rate=0.01,
        fee_basis_points=10,
    ),
    Bridge(
        name="Arbitrum Exit",
        source_chain="Arbitrum",
        target_chain="Ethereum",
        capacity=5000.0,
        latency_mean=86400.0,
        latency_std=3600.0,
        failure_rate=0.005,
        fee_basis_points=10,
    ),
]

# Define pools
pools = {
    "Ethereum": [
        Pool(
            token_a="ETH",
            token_b="USDC",
            reserve_a=1000.0,
            reserve_b=2000000.0,
            fee_tier=30,
        )
    ],
    "Arbitrum": [
        Pool(
            token_a="ETH",
            token_b="USDC",
            reserve_a=500.0,
            reserve_b=1000000.0,
            fee_tier=30,
        )
    ],
}

# Create environment
env = CrossChainEnv(
    chains=chains,
    bridges=bridges,
    pools=pools,
    initial_balances={"ETH": 10.0, "USDC": 20000.0},
    max_steps=100,
)

print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

## Run Random Episode

In [ ]:
# Reset environment
obs, info = env.reset(seed=42)

# Run episode
rewards = []
gas_prices_eth = []
gas_prices_arb = []

for step in range(50):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    rewards.append(reward)
    gas_prices_eth.append(env.chains["Ethereum"].current_gas_price)
    gas_prices_arb.append(env.chains["Arbitrum"].current_gas_price)

    if terminated or truncated:
        break

print(f"Episode length: {len(rewards)}")
print(f"Total reward: {sum(rewards):.4f}")

## Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Rewards
axes[0, 0].plot(rewards)
axes[0, 0].set_title("Step Rewards")
axes[0, 0].set_xlabel("Step")
axes[0, 0].set_ylabel("Reward")
axes[0, 0].grid(True)

# Cumulative rewards
axes[0, 1].plot(np.cumsum(rewards))
axes[0, 1].set_title("Cumulative Reward")
axes[0, 1].set_xlabel("Step")
axes[0, 1].set_ylabel("Cumulative Reward")
axes[0, 1].grid(True)

# Gas prices
axes[1, 0].plot(gas_prices_eth, label="Ethereum")
axes[1, 0].plot(gas_prices_arb, label="Arbitrum")
axes[1, 0].set_title("Gas Prices")
axes[1, 0].set_xlabel("Step")
axes[1, 0].set_ylabel("Gas Price (gwei)")
axes[1, 0].legend()
axes[1, 0].grid(True)

# Final state
stats = env.get_stats()
axes[1, 1].bar(["Swaps", "Bridges"], [stats["num_swaps"], stats["num_bridges"]])
axes[1, 1].set_title("Transaction Counts")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Pool Dynamics

In [ ]:
# Reset and track pool reserves
env.reset(seed=42)

eth_reserves_a = [env.pools["Ethereum"][0].reserve_a]
eth_reserves_b = [env.pools["Ethereum"][0].reserve_b]
arb_reserves_a = [env.pools["Arbitrum"][0].reserve_a]
arb_reserves_b = [env.pools["Arbitrum"][0].reserve_b]

for step in range(30):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    eth_reserves_a.append(env.pools["Ethereum"][0].reserve_a)
    eth_reserves_b.append(env.pools["Ethereum"][0].reserve_b)
    arb_reserves_a.append(env.pools["Arbitrum"][0].reserve_a)
    arb_reserves_b.append(env.pools["Arbitrum"][0].reserve_b)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(eth_reserves_a, label="ETH")
axes[0].plot(np.array(eth_reserves_b) / 2000, label="USDC (scaled)")
axes[0].set_title("Ethereum Pool Reserves")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Reserve Amount")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(arb_reserves_a, label="ETH")
axes[1].plot(np.array(arb_reserves_b) / 2000, label="USDC (scaled)")
axes[1].set_title("Arbitrum Pool Reserves")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Reserve Amount")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Bridge Status Over Time

In [ ]:
# Reset and track bridge loads
env.reset(seed=42)

bridge_loads = [[] for _ in env.bridges]
bridge_statuses = [[] for _ in env.bridges]

for step in range(100):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    for i, bridge in enumerate(env.bridges):
        bridge_loads[i].append(bridge.current_load)
        status_val = {"active": 1, "congested": 0.5, "failed": 0, "maintenance": 0.25}[
            bridge.status.value
        ]
        bridge_statuses[i].append(status_val)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

for i, bridge in enumerate(env.bridges):
    axes[0].plot(bridge_loads[i], label=bridge.name)
axes[0].set_title("Bridge Loads")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Load")
axes[0].legend()
axes[0].grid(True)

for i, bridge in enumerate(env.bridges):
    axes[1].plot(bridge_statuses[i], label=bridge.name, alpha=0.7)
axes[1].set_title("Bridge Status (1=active, 0.5=congested, 0=failed)")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Status")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()